# Distance Analysis Between Validation and Lichess Positions

Sample 1,000 random positions from the validation set, assign 100 random positions from Lichess 20M to each one, and compute the resulting 100,000 move-distance values.

## 1. Import Libraries

In [ ]:
import os, ast, sys
import numpy as np
import pandas as pd
from scipy import stats
from collections import defaultdict
from tqdm.notebook import tqdm

# Add the project root to path to import from train_positions.
sys.path.insert(0, os.path.abspath(".."))
from scipy.optimize import linear_sum_assignment

print("Libraries loaded successfully.")

## 2. Distance Functions (copied from train_positions.py)

In [ ]:
from typing import List, Tuple, Any

def board_matrix_to_pieces(board_matrix: np.ndarray) -> dict:
    piece_channels = {
        0: 'P', 1: 'N', 2: 'B', 3: 'R', 4: 'Q', 5: 'K',
        6: 'p', 7: 'n', 8: 'b', 9: 'r', 10: 'q', 11: 'k'
    }
    pieces = {'white': [], 'black': []}
    for channel, piece_type in piece_channels.items():
        positions = np.argwhere(board_matrix[channel] == 1)
        color = 'white' if channel < 6 else 'black'
        for row, col in positions:
            pieces[color].append((piece_type.upper(), int(row), int(col)))
    return pieces

def knight_distance(r1, c1, r2, c2):
    if (r1, c1) == (r2, c2): return 0
    visited = set()
    queue = [(r1, c1, 0)]
    visited.add((r1, c1))
    while queue:
        r, c, dist = queue.pop(0)
        for dr, dc in [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
            nr, nc = r+dr, c+dc
            if 0<=nr<8 and 0<=nc<8 and (nr,nc) not in visited:
                if (nr,nc)==(r2,c2): return dist+1
                visited.add((nr,nc))
                queue.append((nr,nc,dist+1))
    return 100

def king_distance(r1,c1,r2,c2): return max(abs(r1-r2), abs(c1-c2))

def queen_distance(r1,c1,r2,c2):
    if (r1,c1)==(r2,c2): return 0
    if r1==r2 or c1==c2 or abs(r1-r2)==abs(c1-c2): return 1
    return 2

def rook_distance(r1,c1,r2,c2):
    if (r1,c1)==(r2,c2): return 0
    if r1==r2 or c1==c2: return 1
    return 2

def bishop_distance(r1,c1,r2,c2):
    if (r1,c1)==(r2,c2): return 0
    if abs(r1-r2)==abs(c1-c2): return 1
    if (r1+c1)%2==(r2+c2)%2: return 2
    return 100

def pawn_distance(r1,c1,r2,c2):
    if (r1,c1)==(r2,c2): return 0
    return abs(r1-r2)+abs(c1-c2)

def piece_movement_cost(piece_type, r1, c1, r2, c2):
    pt = piece_type.upper()
    if pt=='K': return king_distance(r1,c1,r2,c2)
    if pt=='Q': return queen_distance(r1,c1,r2,c2)
    if pt=='R': return rook_distance(r1,c1,r2,c2)
    if pt=='B': return bishop_distance(r1,c1,r2,c2)
    if pt=='N': return knight_distance(r1,c1,r2,c2)
    if pt=='P': return pawn_distance(r1,c1,r2,c2)
    return 0

def get_piece_penalty(piece_type):
    return {'Q':10,'R':8,'B':7,'N':6,'P':5,'K':0}.get(piece_type.upper(), 0)

def optimal_assignment_cost(pieces1, pieces2, piece_type):
    if not pieces1 and not pieces2: return 0
    penalty = get_piece_penalty(piece_type)
    m, n = len(pieces1), len(pieces2)
    size = max(m, n)
    cost = np.full((size, size), penalty, dtype=np.float32)
    for i in range(m):
        for j in range(n):
            cost[i,j] = piece_movement_cost(piece_type, pieces1[i][1], pieces1[i][2], pieces2[j][1], pieces2[j][2])
    ri, ci = linear_sum_assignment(cost)
    return int(cost[ri,ci].sum())

def calculate_position_distance(board1: np.ndarray, board2: np.ndarray) -> int:
    p1 = board_matrix_to_pieces(board1)
    p2 = board_matrix_to_pieces(board2)
    total = 0
    for color in ['white', 'black']:
        by_type1 = defaultdict(list)
        by_type2 = defaultdict(list)
        for pt, r, c in p1[color]: by_type1[pt].append((pt,r,c))
        for pt, r, c in p2[color]: by_type2[pt].append((pt,r,c))

        for pt in ['K','Q','R','N','P']:
            l1, l2 = by_type1.get(pt,[]), by_type2.get(pt,[])
            if pt in ('K','Q'):
                if l1 and l2:
                    total += piece_movement_cost(pt, l1[0][1], l1[0][2], l2[0][1], l2[0][2])
                elif len(l1) != len(l2):
                    total += get_piece_penalty(pt)
            else:
                if l1 or l2:
                    total += optimal_assignment_cost(l1, l2, pt)

        # Bishops separated by square color
        b1 = by_type1.get('B',[])
        b2 = by_type2.get('B',[])
        for parity in (0, 1):
            wb1 = [x for x in b1 if (x[1]+x[2])%2==parity]
            wb2 = [x for x in b2 if (x[1]+x[2])%2==parity]
            if wb1 and wb2:
                total += piece_movement_cost('B', wb1[0][1], wb1[0][2], wb2[0][1], wb2[0][2])
            elif len(wb1) != len(wb2):
                total += get_piece_penalty('B')
    return total

def parse_board_matrix(raw) -> np.ndarray:
    """Convert string/list input to a (12,8,8) array (one-hot or piece-grid formats)."""
    piece_to_channel = {
        'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
        'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11
    }

    def _grid_to_one_hot(grid):
        one_hot = np.zeros((12, 8, 8), dtype=np.float32)
        if len(grid) != 8 or any(len(row) != 8 for row in grid):
            raise ValueError(f"Invalid board format: expected 8x8, received {np.array(grid, dtype=object).shape}")
        for r in range(8):
            for c in range(8):
                cell = grid[r][c]
                if isinstance(cell, str):
                    cell = cell.strip()
                if cell in ('.', '', None):
                    continue
                if cell not in piece_to_channel:
                    raise ValueError(f"Unknown piece symbol in board_matrix: {cell!r}")
                one_hot[piece_to_channel[cell], r, c] = 1.0
        return one_hot

    if isinstance(raw, np.ndarray):
        arr = raw
    elif isinstance(raw, str):
        arr = ast.literal_eval(raw)
    else:
        arr = raw

    arr_np = np.array(arr, dtype=object)

    if arr_np.shape == (12, 8, 8):
        return np.array(arr, dtype=np.float32)
    if arr_np.shape == (8, 8, 12):
        return np.transpose(np.array(arr, dtype=np.float32), (2, 0, 1)).astype(np.float32)
    if arr_np.shape == (8, 8):
        return _grid_to_one_hot(arr)

    raise ValueError(f"Unsupported board_matrix format: {arr_np.shape}")

print("Distance functions defined successfully.")

## 3. Load Datasets

In [ ]:
VAL_CSV = os.path.join("data", "final", "val_df_final.csv")
LICHESS_CANDIDATES = [
    os.path.join("data", "clean", "lichess_cleaned_minmoves6.csv"),
    os.path.join("data", "clean", "lichess_cleaned.csv"),
    os.path.join("datasets", "dataset_lichess_db_20M.csv"),
]

LICHESS_CSV = None
for candidate in LICHESS_CANDIDATES:
    if not os.path.exists(candidate):
        continue
    cols = list(pd.read_csv(candidate, nrows=0).columns)
    if "board_matrix" in cols:
        LICHESS_CSV = candidate
        break

if LICHESS_CSV is None:
    raise ValueError("No Lichess CSV with a 'board_matrix' column was found. Check LICHESS_CANDIDATES.")

val_df = pd.read_csv(VAL_CSV)
print(f"Validation set: {val_df.shape[0]:,} rows, columns: {list(val_df.columns)}")
print(f"Selected Lichess CSV: {LICHESS_CSV}")

# Read just a quick preview of the validation dataset.
print("\nFirst rows of the validation dataset:")
val_df.head(3)

## 4. Sample 1,000 positions from the validation dataset

In [ ]:
SEED = 42
N_VAL  = 1_000   # validation positions to sample
N_NEG  = 100     # negative positions per validation position

val_sample = val_df.sample(n=min(N_VAL, len(val_df)), random_state=SEED).reset_index(drop=True)
print(f"Sampled {len(val_sample):,} validation positions.")
val_sample[["board_matrix"]].head(2)

## 5. Sample 100 x 1,000 = 100,000 Lichess positions

Sample 100,000 Lichess rows in a single pass (much faster than 1,000 separate reads) and split them into 1,000 groups of 100.

In [ ]:
TOTAL_NEG = N_VAL * N_NEG   # 100 000

# Count rows in the Lichess CSV without loading the whole file.
print("Counting rows in the Lichess CSV (this may take a while)...")
lichess_nrows = sum(1 for _ in open(LICHESS_CSV, encoding="utf-8")) - 1  # subtract header
print(f"  -> {lichess_nrows:,} rows in {LICHESS_CSV}")

# Generate row indices to read (0-based, excluding header).
rng = np.random.default_rng(SEED)
skip_rows = sorted(rng.choice(lichess_nrows, size=TOTAL_NEG, replace=False))

# Read only sampled rows using skiprows (always keep header line 0).
skip_set = set(skip_rows)
skipfunc = lambda i: i > 0 and (i - 1) not in skip_set   # i is the file line number

print(f"Reading {TOTAL_NEG:,} Lichess rows... (this may take several minutes)")
lichess_sample = pd.read_csv(LICHESS_CSV, skiprows=skipfunc)
lichess_sample = lichess_sample.reset_index(drop=True)
print(f"Read {len(lichess_sample):,} rows. Columns: {list(lichess_sample.columns)}")
lichess_sample.head(2)

## 6. Compute the 100,000 distances

In [ ]:
# Pre-parse validation board_matrix values.
print("Parsing validation positions...")
val_boards = [parse_board_matrix(row) for row in tqdm(val_sample["board_matrix"], leave=False)]

# Pre-parse Lichess board_matrix values.
print("Parsing Lichess positions...")
lich_boards = [parse_board_matrix(row) for row in tqdm(lichess_sample["board_matrix"], leave=False)]

n_val_eff = len(val_boards)
required_neg = n_val_eff * N_NEG
if len(lich_boards) < required_neg:
    raise ValueError(f"Not enough Lichess positions: {len(lich_boards):,} < {required_neg:,}.")

# Compute distances.
all_distances = np.empty(required_neg, dtype=np.int32)

print(f"\nComputing {required_neg:,} distances...")
idx = 0
for i in tqdm(range(n_val_eff), desc="Validation positions"):
    val_b = val_boards[i]
    for j in range(N_NEG):
        all_distances[idx] = calculate_position_distance(val_b, lich_boards[i * N_NEG + j])
        idx += 1

print(f"\nDistances computed: {len(all_distances):,}")

## 7. Save distances and compute summary statistics

In [ ]:
# Save as .npy for later use.
save_path = "distances_val1k_lichess100.npy"
np.save(save_path, all_distances)
print(f"Distances saved to: {save_path}")

# Statistics
mean_d   = float(np.mean(all_distances))
median_d = float(np.median(all_distances))
mode_res = stats.mode(all_distances, keepdims=True)
mode_d   = int(mode_res.mode[0])
mode_cnt = int(mode_res.count[0])

print(f"\n{'='*40}")
print(f"  Total distances       : {len(all_distances):,}")
print(f"  Mean                  : {mean_d:.4f}")
print(f"  Median                : {median_d:.1f}")
print(f"  Mode                  : {mode_d} (appears {mode_cnt:,} times)")
print(f"  Min                   : {int(all_distances.min())}")
print(f"  Max                   : {int(all_distances.max())}")
print(f"  Std. dev.             : {float(np.std(all_distances)):.4f}")
print(f"{'='*40}")

## 8. Histogram of distance distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full histogram
axes[0].hist(all_distances, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(mean_d,   color="red",    linestyle="--", linewidth=1.5, label=f"Mean={mean_d:.1f}")
axes[0].axvline(median_d, color="orange", linestyle="--", linewidth=1.5, label=f"Median={median_d:.1f}")
axes[0].axvline(mode_d,   color="green",  linestyle="--", linewidth=1.5, label=f"Mode={mode_d}")
axes[0].set_title("Distribution of 100,000 Distances")
axes[0].set_xlabel("Distance in Moves")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Zoom: 5th to 95th percentile
p5, p95 = np.percentile(all_distances, 5), np.percentile(all_distances, 95)
zoom_data = all_distances[(all_distances >= p5) & (all_distances <= p95)]
axes[1].hist(zoom_data, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].axvline(mean_d,   color="red",    linestyle="--", linewidth=1.5, label=f"Mean={mean_d:.1f}")
axes[1].axvline(median_d, color="orange", linestyle="--", linewidth=1.5, label=f"Median={median_d:.1f}")
axes[1].axvline(mode_d,   color="green",  linestyle="--", linewidth=1.5, label=f"Mode={mode_d}")
axes[1].set_title("Zoom (5th-95th Percentile)")
axes[1].set_xlabel("Distance in Moves")
axes[1].legend()

plt.tight_layout()
plt.savefig("distances_histogram.png", dpi=150)
plt.show()
print("Histogram saved to: distances_histogram.png")

In [ ]:
import chess
from IPython.display import display, HTML

def board_matrix_to_chess_board(board_matrix):
    """Convert board_matrix into a chess.Board object."""
    board = chess.Board.empty()
    pieces = board_matrix_to_pieces(board_matrix)
    for color, piece_list in pieces.items():
        for piece_type, r, c in piece_list:
            square = chess.square(c, 7 - r)
            symbol = piece_type.lower() if color == 'black' else piece_type.upper()
            piece = chess.Piece.from_symbol(symbol)
            board.set_piece_at(square, piece)
    return board

# Specify the target distance.
target_distance = 125  # Change this value as needed.

# Find indices where distance matches the target.
matching_indices = np.where(all_distances == target_distance)[0]
if len(matching_indices) == 0:
    print(f"No pairs found with distance {target_distance}.")
else:
    # Take the first matching pair.
    idx = matching_indices[0]
    i = idx // N_NEG  # index in val_sample
    j = idx % N_NEG   # index in the Lichess group for that validation sample
    
    val_board = val_boards[i]
    lich_board = lich_boards[i * N_NEG + j]
    
    # Convert to chess.Board objects.
    val_chess_board = board_matrix_to_chess_board(val_board)
    lich_chess_board = board_matrix_to_chess_board(lich_board)
    
    # Generate SVGs.
    svg1 = val_chess_board._repr_svg_()
    svg2 = lich_chess_board._repr_svg_()
    
    # Display side by side using HTML.
    html = f"""
    <div style="display: flex; justify-content: space-around; align-items: center;">
        <div style="text-align: center;">
            <h3>Validation Position (index {i})<br>Distance: {target_distance}</h3>
            {svg1}
        </div>
        <div style="text-align: center;">
            <h3>Lichess Position (index {i*N_NEG + j})<br>Distance: {target_distance}</h3>
            {svg2}
        </div>
    </div>
    """
    display(HTML(html))
    
    print(f"Showing positions with distance {target_distance}: val[{i}] and lich[{i*N_NEG + j}]")